# 3D Gaussian Splatting & Projective Rendering Recipe using `algebrax`

**Mathematical Foundations & Theory:**
1. **3D Spatial Covariance ($\\Sigma = R S S^T R^T$)**: Constructed using `algebrax.matrix.core.dot` and `transpose` over $SO(3)$ rotation and scaling matrices.
2. **2D Perspective Screen Projection ($\\Sigma' = J W \\Sigma W^T J^T$)**: Evaluates Jacobian projective transformation $J$ using matrix multiplication.
3. **Depth-Sorted Volumetric $\\alpha$-Compositing**: Ray-marching $\\alpha$-blending accumulated along depth-sorted 3D Gaussians with `algebrax.analysis.gaussian_kernel`.

In [ ]:
import math

import algebrax as ax


def create_scale_matrix(sx: float, sy: float, sz: float) -> dict[int, dict[int, float]]:
    return {0: {0: sx}, 1: {1: sy}, 2: {2: sz}}

def create_rotation_matrix(pitch: float, yaw: float, roll: float) -> dict[int, dict[int, float]]:
    cx, sx = math.cos(pitch), math.sin(pitch)
    cy, sy = math.cos(yaw), math.sin(yaw)
    cz, sz = math.cos(roll), math.sin(roll)
    rx = {0: {0: 1.0}, 1: {1: cx, 2: -sx}, 2: {1: sx, 2: cx}}
    ry = {0: {0: cy, 2: sy}, 1: {1: 1.0}, 2: {0: -sy, 2: cy}}
    rz = {0: {0: cz, 1: -sz}, 1: {0: sz, 1: cz}, 2: {2: 1.0}}
    return ax.matrix.dot(rz, ax.matrix.dot(ry, rx))

def compute_3d_covariance(scale, rot):
    s_mat = create_scale_matrix(*scale)
    r_mat = create_rotation_matrix(*rot)
    s_sq = ax.matrix.dot(s_mat, s_mat)
    return ax.matrix.dot(ax.matrix.dot(r_mat, s_sq), ax.matrix.transpose(r_mat))

def compute_2d_projected_covariance(sigma_3d, mean_3d, focal_length=2.5):
    tx, ty, tz = mean_3d
    tz = max(tz, 0.1)
    j_mat = {
        0: {0: focal_length / tz, 2: -focal_length * tx / (tz * tz)},
        1: {1: focal_length / tz, 2: -focal_length * ty / (tz * tz)},
    }
    return ax.matrix.dot(ax.matrix.dot(j_mat, sigma_3d), ax.matrix.transpose(j_mat))


In [ ]:
# Execute 3D Covariance and 2D Projection Pipeline

gaussians = [
    {
        "id": "Red_Splat",
        "pos": (0.0, 0.0, 4.0),
        "scale": (0.8, 0.3, 0.3),
        "rot": (0.2, 0.5, 0.0),
        "color": (1.0, 0.2, 0.2),
        "opacity": 0.85
    },
    {
        "id": "Blue_Splat",
        "pos": (0.5, 0.3, 3.5),
        "scale": (0.4, 0.7, 0.4),
        "rot": (0.0, -0.3, 0.4),
        "color": (0.2, 0.4, 1.0),
        "opacity": 0.75
    },
]

for g in gaussians:
    g["sigma_3d"] = compute_3d_covariance(g["scale"], g["rot"])
    g["sigma_2d"] = compute_2d_projected_covariance(g["sigma_3d"], g["pos"])
    print(f"--- {g['id']} ---")
    print("3D Covariance Sigma:", g["sigma_3d"])
    print("2D Projected Screen Sigma':", g["sigma_2d"])
